# Modelo ML XGBOOST CLASSIFIER

In [3]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score
import numpy as np
import pandas as pd


train_data = pd.read_csv("../models/x_train_sel.csv")
test_data = pd.read_csv("../models/x_test_sel.csv")


X_train = train_data.drop(["ciudad"], axis = 1)
y_train = train_data["ciudad"]
X_test = test_data.drop(["ciudad"], axis = 1)
y_test = test_data["ciudad"]

X_train.head()
y_train.head()
X_test.head()
y_test.head()


0    45
1    60
2     7
3     5
4    43
Name: ciudad, dtype: int64

In [4]:

# 4. Entrenar modelo
model = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
model.fit(X_train, y_train)

# 5. Evaluar
y_pred = model.predict(X_test)
print("✔️ Accuracy:", accuracy_score(y_test, y_pred))
print("\n📊 Clasificación:\n", classification_report(y_test, y_pred))

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/xgboost/core.py:158: UserWarning: [03:53:02] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


✔️ Accuracy: 1.0

📊 Clasificación:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       818
           1       1.00      1.00      1.00       789
           2       1.00      1.00      1.00       415
           3       1.00      1.00      1.00         2
           4       1.00      1.00      1.00        38
           5       1.00      1.00      1.00       827
           6       1.00      1.00      1.00       491
           7       1.00      1.00      1.00       712
           8       1.00      1.00      1.00       682
           9       1.00      1.00      1.00        20
          10       1.00      1.00      1.00       617
          11       1.00      1.00      1.00       802
          12       1.00      1.00      1.00       513
          13       1.00      1.00      1.00       633
          14       1.00      1.00      1.00        38
          15       1.00      1.00      1.00        29
          16       1.00      1.00      1.00  

# 🧪 PREDICCIÓN PERSONALIZADA

In [42]:
X_train.columns.to_frame().to_csv("../data/processed/x_train_columns.csv", index=False, header=False)

# PASO 2: Definir función para construir entrada de usuario

import pandas as pd

def construir_input_usuario(valores_activados, columnas_referencia_path="x_train_columns.csv"):
    """
    Construye un DataFrame con la estructura del modelo, activando las columnas que el usuario seleccionó.
    """
    columnas_modelo = pd.read_csv(columnas_referencia_path, header=None).squeeze().tolist()
    
    # Crear un DataFrame vacío con todas las columnas a 0
    input_df = pd.DataFrame(columns=columnas_modelo)
    input_df.loc[0] = 0

    # Activar las columnas que el usuario seleccionó
    for col, val in valores_activados.items():
        if col in input_df.columns:
            input_df.at[0, col] = val
    return input_df

# PASO 3: Simular selección del usuario
seleccion_usuario = {
    "perfil_Cultura": 1,
    "perfil_Relax": 1,
    "entorno_Ciudad": 1,
    "temporada_Verano": 1,
    "origin_city_Madrid": 1
}

X_user = construir_input_usuario(
    seleccion_usuario,
    columnas_referencia_path="../data/processed/x_train_columns.csv"
)


In [51]:
import json

with open("../data/processed/Json/ciudad_transformation_rules.json", "r") as f:
    ciudad_mapping = json.load(f)

# Invertir el diccionario: de ID a nombre
id_to_ciudad = {str(v): k for k, v in ciudad_mapping.items()}

In [53]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Cargar
y_train = pd.read_excel("../data/processed/X&Ys/y_train.xlsx").squeeze()

# Convertir de ID a nombre
y_train = y_train.astype(str).map(id_to_ciudad)

# Comprobar los primeros valores
print(y_train.head())

0    amsterdam
1       monaco
2        túnez
3       mumbai
4    melbourne
Name: ciudad_n, dtype: object


In [54]:
le = LabelEncoder()
le.fit(y_train)

LabelEncoder()

In [57]:
probs = model.predict_proba(X_user)[0]
top5_indices = np.argsort(probs)[::-1][:5]
top5_labels = model.classes_[top5_indices]
top5_ciudades = le.inverse_transform(top5_labels)

print("🏝️ Top 5 ciudades recomendadas:", top5_ciudades)

🏝️ Top 5 ciudades recomendadas: ['reykjavik' 'amman' 'perth' 'vilnius' 'marrakech']


# 🌍 FUNCIONES DE ENRIQUECIMIENTO (usando df original)

In [ ]:
def get_clima_estimado(ciudad, temporada):
    clima = df[(df['ciudad'] == ciudad) & (df['temporada'] == temporada)]
    if clima.empty:
        return "Sin datos"
    return clima[['temp_max', 'temp_min', 'precipitacion']].mean().round(1).to_dict()

def get_eventos(ciudad):
    eventos = df[df['ciudad'] == ciudad][['evento_nombre', 'evento_categoria', 'evento_desc', 'fecha']]
    eventos = eventos.dropna().drop_duplicates().head(3)
    return eventos.to_dict(orient='records') if not eventos.empty else "Sin eventos"

def get_precio_vuelo(origen, destino):
    vuelos = df[(df['origin_city'] == origen) & (df['ciudad'] == destino)]
    return round(vuelos['flight_price'].mean(), 2) if not vuelos.empty else "Sin datos"

def get_hotel(ciudad):
    hoteles = df[df['ciudad'] == ciudad][['hotel_name', 'estimated_price_eur_y', 'hotel_type', 'distance_to_city_center_km']]
    hotel = hoteles.dropna().sort_values(by='estimated_price_eur_y').head(1)
    return hotel.to_dict(orient='records')[0] if not hotel.empty else "Sin hoteles"

# 📦 MOSTRAR INFO ENRIQUECIDA

In [ ]:
# Suponiendo que defines estas variables desde la selección de usuario:
temporada_usuario = 'Verano'
origen_usuario = 'Madrid'

# Mostrar la información para las 3 ciudades
for ciudad in top3_ciudades:
    print(f"\n🌍 Ciudad: {ciudad}")
    print("☁️ Clima estimado:", get_clima_estimado(ciudad, temporada_usuario))
    print("🎫 Eventos:", get_eventos(ciudad))
    print("✈️ Vuelo desde origen:", get_precio_vuelo(origen_usuario, ciudad))
    print("🏨 Hotel recomendado:", get_hotel(ciudad))